### Exercise 1

We can explore the connection between exponential families and softmax in some more depth.
1. Compute the second derivative of the cross-entropy loss $l(\mathbf{y}, \hat{\mathbf{y}})$ for softmax.
2. Compute the variance of the distribution given by soft $\max (\mathbf{o})$ and show that it matches the second derivative computed above.

Let's do this for a single example first (one row of the minibatch). I'll write the logits as $0 \in \mathbb{R}^q$, softmax probabilities as
$$
\mathbf{p}=\operatorname{softmax}(\mathbf{o}), \quad p_i=\frac{e^{o_i}}{\sum_{k=1}^q e^{o_k}},
$$
and a one-hot label $\mathbf{y} \in\{0,1\}^q$ with $\sum_i y_i=1$.

The (per-example) cross-entropy loss is
$$
\ell(\mathbf{y}, \mathbf{p})=-\sum_{i=1}^q y_i \log p_i .
$$

Step A: softmax Jacobian

Differentiate $p_i$ w.r.t. $o_j$ :
$$
\frac{\partial p_i}{\partial o_j}=p_i\left(\delta_{i j}-p_j\right),
$$
where $\delta_{i j}$ is 1 if $i=j$, else 0 .

In matrix form, the Jacobian $J \in \mathbb{R}^{q \times q}$ of $\mathbf{p}$ w.r.t. $\mathbf{o}$ is
$$
J=\frac{\partial \mathbf{p}}{\partial \mathrm{o}}=\operatorname{diag}(\mathbf{p})-\mathbf{p} \mathbf{p}^{\top} .
$$

Step B: gradient of cross-entropy w.r.t. logits

Use $\ell=-\sum_i y_i \log p_i$. First:
$$
\frac{\partial \ell}{\partial p_i}=-\frac{y_i}{p_i} .
$$
Chain rule:
$$
\frac{\partial \ell}{\partial o_j}=\sum_{i=1}^q \frac{\partial \ell}{\partial p_i} \frac{\partial p_i}{\partial o_j}=\sum_i\left(-\frac{y_i}{p_i}\right) p_i\left(\delta_{i j}-p_j\right)=-\sum_i y_i\left(\delta_{i j}-p_j\right) .
$$
Since $\sum_i y_i=1$, this becomes
$$
\frac{\partial \ell}{\partial o_j}=p_j-y_j .
$$
So in vector form:
$$
\nabla_{\mathrm{o}} \ell=\mathbf{p}-\mathbf{y} .
$$

Step C: Hessian w.r.t. logits

Differentiate the gradient again:
$$
\nabla_{\mathrm{o}}^2 \ell=\frac{\partial(\mathbf{p}-\mathbf{y})}{\partial \mathrm{o}}=\frac{\partial \mathbf{p}}{\partial \mathrm{o}}=\operatorname{diag}(\mathbf{p})-\mathbf{p p}^{\top} .
$$
So the second derivative (Hessian) is:
$$
\nabla_{\mathrm{o}}^2 \ell=\operatorname{diag}(\mathrm{p})-\mathbf{p p}^{\top}
$$

Interpret $\mathbf{p}=\operatorname{softmax}(\mathbf{o})$ as defining a categorical distribution over $q$ classes.

A clean way to match the Hessian is to look at the random one-hot vector
$$
\mathbf{Z} \in\left\{\mathbf{e}_1, \ldots, \mathbf{e}_q\right\}, \quad \mathbb{P}\left(\mathbf{Z}=\mathbf{e}_i\right)=p_i,
$$
where $\mathbf{e}_i$ is the $i$-th standard basis vector.

Mean
$$
\mathbb{E}[\mathbf{Z}]=\sum_{i=1}^q p_i \mathrm{e}_i=\mathrm{p} .
$$
Second moment
$$
\mathrm{ZZ}^{\top}=\left\{\mathrm{e}_i \mathrm{e}_i^{\top} \quad \text { with prob } p_i,\right.
$$
so
$$
\mathbb{E}\left[\mathbf{Z Z}^{\top}\right]=\sum_{i=1}^q p_i \mathbf{e}_i \mathbf{e}_i^{\top}=\operatorname{diag}(\mathbf{p}) .
$$
Covariance ("variance matrix")
$$
\operatorname{Cov}(\mathbf{Z})=\mathbb{E}\left[\mathbf{Z} \mathbf{Z}^{\top}\right]-\mathbb{E}[\mathbf{Z}] \mathbb{E}[\mathbf{Z}]^{\top}=\operatorname{diag}(\mathbf{p})-\mathbf{p p}^{\top} .
$$

### Exercise 2

Assume that we have three classes which occur with equal probability, i.e., the probability vector is $\left(\frac{1}{3}, \frac{1}{3}, \frac{1}{3}\right)$.
1. What is the problem if we try to design a binary code for it?
2. Can you design a better code? Hint: what happens if we try to encode two independent observations? What if we encode $n$ observations jointly?

#### 2.1

If we want a fixed-length binary code (same number of bits every time), then:
- 1 bit gives only 2 codewords → not enough for 3 outcomes.
- So we need at least 2 bits $(00,01,10,11) \rightarrow$ gives 4 codewords, but you only need 3 .

So with 2-bit fixed-length coding, we "waste" one codeword and the expected length is 2 bits/symbol.

But the entropy of the source is
$$
H=\log _2 3 \approx 1.585 \text { bits/symbol },
$$
so 2 bits/symbol is not optimal: you have a gap of about 0.415 bits/symbol.

#### 2.2

Encode 3 observations jointly (a clean improvement with fixed length)

Now $3^3=27$ blocks. Since $2^5=32$,
$$
\left\lceil\log _2 27\right\rceil=5 \text { bits per block. }
$$
Per symbol that's
$$
\frac{5}{3} \approx 1.667 \mathrm{bits} / \mathrm{symbol},
$$
which is closer to 1.585 .

General $n$ : fixed-length block code

There are $3^n$ blocks. Any fixed-length binary code needs
$$
L_n=\left\lceil\log _2\left(3^n\right)\right\rceil=\left\lceil n \log _2 3\right\rceil
$$
bits per block, i.e.
$$
\frac{L_n}{n}=\frac{\left\lceil n \log _2 3\right\rceil}{n} .
$$
As $n \rightarrow \infty$,
$$
\frac{\left\lceil n \log _2 3\right\rceil}{n} \rightarrow \log _2 3,
$$
so the overhead per symbol goes to 0.

### Exercise 4

The Bradley-Terry model uses a logistic model to capture preferences. For a user to choose between apples and oranges one assumes scores $o_{\text {apple }}$ and $o_{\text {orange }}$. Our requirements are that larger scores should lead to a higher likelihood in choosing the associated item and that the item with the largest score is the most likely one to be chosen (Bradley and Terry, 1952).
1. Prove that softmax satisfies this requirement.
2. What happens if you want to allow for a default option of choosing neither apples nor oranges? Hint: now the user has three choices.

#### 4.1

Setup
Two items with scores (logits) $o_a$ and $o_o$. Softmax over two choices gives
$$
P(\text { apple })=\frac{e^{o_a}}{e^{o_a}+e^{o_o}}, \quad P(\text { orange })=\frac{e^{o_o}}{e^{o_a}+e^{o_o}} .
$$
(And note this is exactly the Bradley-Terry / logistic form:
$$
\left.P(\text { apple })=\sigma\left(o_a-o_o\right) .\right)
$$

Requirement A: Larger score ⇒ higher likelihood

Show $P$ (apple) increases with $o_a$ and decreases with $o_o$.

Differentiate:
$$
\begin{aligned}
& \frac{\partial P(\text { apple })}{\partial o_a}=P(\text { apple })(1-P(\text { apple }))>0, \\
& \frac{\partial P(\text { apple })}{\partial o_o}=-P(\text { apple }) P(\text { orange })<0 .
\end{aligned}
$$
So increasing apple's score raises its choice probability, and increasing orange's score lowers apple's probability. Symmetrically for orange.

Requirement B: Largest score is most likely chosen

Compare probabilities:
$$
\frac{P(\text { apple })}{P(\text { orange })}=\frac{e^{o_a}}{e^{o_o}}=e^{o_a-o_o} .
$$
So
- $o_a>o_o \Longleftrightarrow e^{o_a-o_o}>1 \Longleftrightarrow P$ (apple) $>P$ (orange).
- If $o_a=o_o$, then $P($ apple $)=P($ orange $)=1 / 2$.

Thus the item with larger score has larger selection probability (and hence is most likely).

#### 4.2

Add a third option and apply softmax over three logits:
$$
P(\text { apple })=\frac{e^{o_a}}{e^{o_a}+e^{o_o}+e^{o_0}}, \quad P(\text { orange })=\frac{e^{o_o}}{e^{o_a}+e^{o_o}+e^{o_0}}, \quad P(\text { neither })=\frac{e^{o_0}}{e^{o_a}+e^{o_o}+e^{o_0}} .
$$
What changes / what "happens" conceptually?
- Apple is still more likely than orange iff $o_a>o_o$, because
$$
\frac{P(\text { apple })}{P(\text { orange })}=e^{o_a-o_o}
$$
(the $o_0$ term cancels).
- But now the most likely choice among the three is whichever has the largest logit among $\left\{o_a, o_o, o_0\right\}$

### Exercise 5

Softmax gets its name from the following mapping: $\operatorname{RealSoftMax}(a, b)=\log (\exp (a)+\exp (b))$.
1. Prove that $\operatorname{RealSoftMax}(a, b)>\max (a, b)$.
2. How small can you make the difference between both functions? Hint: without loss of generality you can set $b=0$ and $a \geq b$.
3. Prove that this holds for $\lambda^{-1} \operatorname{RealSoftMax}(\lambda a, \lambda b)$, provided that $\lambda>0$.
4. Show that for $\lambda \rightarrow \infty$ we have $\lambda^{-1} \operatorname{RealSoftMax}(\lambda a, \lambda b) \rightarrow \max (a, b)$.
5. Construct an analogous softmin function.
6. Extend this to more than two numbers.

Let
$$
\operatorname{RSM}(a, b)=\log \left(e^a+e^b\right) .
$$
WLOG assume $a \geq b$. Then $\max (a, b)=a$.

A super useful rewrite is:
$$
\operatorname{RSM}(a, b)=\log \left(e^a\left(1+e^{b-a}\right)\right)=a+\log \left(1+e^{b-a}\right) .
$$
Let $\Delta=a-b \geq 0$. Then $b-a=-\Delta \leq 0$, so
$$
\operatorname{RSM}(a, b)=a+\log \left(1+e^{-\Delta}\right) .
$$

#### 5.1

Since $a \geq b$, $\max$ is $a$. And because $e^{-\Delta}>0$,
$$
\log \left(1+e^{-\Delta}\right)>0 .
$$
So
$$
\operatorname{RSM}(a, b)=a+\log \left(1+e^{-\Delta}\right)>a=\max (a, b) .
$$

#### 5.2

The difference is
$$
\operatorname{RSM}(a, b)-\max (a, b)=\log \left(1+e^{-\Delta}\right) .
$$
- If $\Delta \rightarrow \infty$ (i.e., a much larger than $b$ ), then $e^{-\Delta} \rightarrow 0$ and
$$
\log \left(1+e^{-\Delta}\right) \rightarrow \log (1)=0
$$
So you can make the difference arbitrarily close to 0 (but not equal to 0 for finite $\Delta$ ).
Also note the largest possible gap occurs at $\Delta=0$ (i.e., $a=b$ ):
$$
\log \left(1+e^0\right)=\log 2 .
$$
So overall:
$$
0<\operatorname{RSM}(a, b)-\max (a, b) \leq \log 2 .
$$

#### 5.3

Define
$$
f_\lambda(a, b)=\frac{1}{\lambda} \log \left(e^{\lambda a}+e^{\lambda b}\right) .
$$
Assume $a \geq b$. Factor out $e^{\lambda a}$ :
$$
f_\lambda(a, b)=\frac{1}{\lambda} \log \left(e^{\lambda a}\left(1+e^{\lambda(b-a)}\right)\right)=a+\frac{1}{\lambda} \log \left(1+e^{-\lambda \Delta}\right) .
$$
So
$$
f_\lambda(a, b)-\max (a, b)=\frac{1}{\lambda} \log \left(1+e^{-\lambda \Delta}\right)>0
$$
and similarly it's bounded by $\frac{\log 2}{\lambda}$ (achieved at $\Delta=0$ ):
$$
0<f_\lambda(a, b)-\max (a, b) \leq \frac{\log 2}{\lambda} .
$$

#### 5.4

$\lim _{\lambda \rightarrow \infty} \frac{1}{\lambda} \log \left(e^{\lambda a}+e^{\lambda b}\right)=\max (a, b)$.

#### 5.5

Min is negative max of negatives:
$$
\min (a, b)=-\max (-a,-b) .
$$
So define
$$
\operatorname{SoftMin}_\lambda(a, b)=-\frac{1}{\lambda} \log \left(e^{-\lambda a}+e^{-\lambda b}\right) .
$$

#### 5.6

For $x_1, \ldots, x_n$, define the softmax ("log-sum-exp") approximation to max:
$$
\operatorname{SoftMax}_\lambda\left(x_1, \ldots, x_n\right)=\frac{1}{\lambda} \log \left(\sum_{i=1}^n e^{\lambda x_i}\right) .
$$
It satisfies
$$
\max _i x_i<\operatorname{SoftMax}_\lambda\left(x_1, \ldots, x_n\right) \leq \max _i x_i+\frac{\log n}{\lambda},
$$
and
$$
\operatorname{SoftMax}_\lambda \underset{\lambda \rightarrow \infty}{\longrightarrow} \max _i x_i \text {. }
$$

### Exercise 6

The function $g(\mathbf{x}) \stackrel{\text { def }}{=} \log \sum_i \exp x_i$ is sometimes also referred to as the log-partition function.
1. Prove that the function is convex. Hint: to do so, use the fact that the first derivative amounts to the probabilities from the softmax function and show that the second derivative is the variance.
2. Show that $g$ is translation invariant, i.e., $g(\mathbf{x}+b)=g(\mathbf{x})$.
3. What happens if some of the coordinates $x_i$ are very large? What happens if they're all very small?
4. Show that if we choose $b=\max _i x_i$ we end up with a numerically stable implementation.

Let
$$
g(\mathbf{x})=\log \sum_{i=1}^q e^{x_i}, \quad \mathbf{x} \in \mathbb{R}^q,
$$
and define $Z=\sum_i e^{x_i}$ and
$$
p_i=\frac{e^{x_i}}{Z} \quad(\text { so } \mathbf{p}=\operatorname{softmax}(\mathbf{x})) .
$$

#### 6.1

First derivative (gradient)

For each coordinate $j$,
$$
\frac{\partial g}{\partial x_j}=\frac{1}{Z} \frac{\partial}{\partial x_j} \sum_i e^{x_i}=\frac{1}{Z} e^{x_j}=p_j .
$$
So
$$
\nabla g(\mathbf{x})=\mathbf{p}=\operatorname{softmax}(\mathbf{x}) .
$$
Second derivative (Hessian)

Differentiate $p_i$ w.r.t. $x_j$ :
$$
\frac{\partial p_i}{\partial x_j}=p_i\left(\delta_{i j}-p_j\right) .
$$
Hence the Hessian is
$$
\nabla^2 g(\mathbf{x})=\operatorname{diag}(\mathbf{p})-\mathbf{p p}^{\top} .
$$
In exercise 1.2, we showed that this is the covariance matrix of the distribution given by softmax(o) and therefore, it is positive semdefinite.

#### 6.2

If $b \in \mathbb{R}$ and $1=(1, \ldots, 1)$,
$$
g(\mathbf{x}+b \mathbf{1})=\log \sum_i e^{x_i+b}=\log \left(e^b \sum_i e^{x_i}\right)=b+\log \sum_i e^{x_i}=b+g(\mathbf{x}) .
$$
So $g$ is translation equivariant (shifts by $b$ ), not invariant.

#### 6.3

Let $m=\max _i x_i$.
- If one coordinate $x_k$ is much larger than the rest (big gaps), then
$$
\sum_i e^{x_i} \approx e^{x_k}, \Rightarrow g(\mathbf{x}) \approx x_k=m,
$$
and softmax concentrates: $p_k \approx 1$, others $\approx 0$.
- If all coordinates are "very small" (i.e., very negative), then each $e^{x_i}$ is tiny and the raw sum can underflow numerically. Mathematically, $g(\mathrm{x})$ is still finite, but numerically $\sum_i e^{x_i}$ might become 0 in floating point, causing $\log 0$ errors.

#### 6.4

Choose $b=m=\max _i x_i$. Then write:
$$
g(\mathbf{x})=\log \sum_i e^{x_i}=\log \left(e^m \sum_i e^{x_i-m}\right)=m+\log \sum_i e^{x_i-m} .
$$
Now every exponent $x_i-m \leq 0$, so $e^{x_i-m} \in(0,1]$. This prevents overflow from huge $e^{x_i}$, and also improves behavior when numbers are very negative.

So the stable formula is:
$$
g(\mathbf{x})=m+\log \sum_i \exp \left(x_i-m\right), \quad m=\max _i x_i .
$$